# 03. Customer & Loan Segmentation (K-Means & PCA)
## Project: AI-Driven Loan Recovery & Risk Analytics

### Overview
This notebook builds unsupervised machine learning clusters to identify distinct borrower personas:
1. **Feature Standardisation**: Standardizing multi-scale financial and behavioral metrics.
2. **Hyperparameter Selection**: Evaluating the Elbow Method (Inertia) and Silhouette Scores for optimal $K$.
3. **Persona Profiling**: Mapping numerical clusters to business personas (Prime Self-Cure, Distressed Responsive, Chronic High Risk, Secured Asset Backed).
4. **Dimensionality Reduction**: Visualizing multi-dimensional clusters using 2D Principal Component Analysis (PCA).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

df = pd.read_csv("../data/processed/loan_recovery_master.csv")
features = ["credit_score", "annual_income", "loan_amount", "interest_rate", "overdue_days", "dti_ratio", "installments_paid", "delinquency_severity_score"]
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Features scaled: {X_scaled.shape}")


### 1. Elbow Method & Silhouette Optimization

In [ ]:
inertias = []
sil_scores = []
k_vals = range(2, 7)

for k in k_vals:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(k_vals, inertias, 'bo-', label="Inertia (Elbow)")
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("Inertia", color='b')

ax2 = ax1.twinx()
ax2.plot(k_vals, sil_scores, 'r^-', label="Silhouette Score")
ax2.set_ylabel("Silhouette Score", color='r')
plt.title("Cluster Evaluation: Elbow & Silhouette Scores", fontweight="bold")
plt.show()


### 2. K-Means Training & Persona Assignment

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=15)
df["cluster_id"] = kmeans.fit_predict(X_scaled)

# PCA Projection
pca = PCA(n_components=2, random_state=42)
pca_res = pca.fit_transform(X_scaled)
df["pca_1"] = pca_res[:, 0]
df["pca_2"] = pca_res[:, 1]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="pca_1", y="pca_2", hue="cluster_id", palette="tab10", alpha=0.6)
plt.title("2D PCA Visualization of Customer Segments", fontweight="bold")
plt.show()
